In [1]:
# ---- Cell 1 code ----
import os
import pandas as pd
import numpy as np

# Forward slashes avoid Windows escape-character errors in string paths
data_dir = 'C:/Users/majid/Desktop/Data Analyst Portfolio/4_Supply_Chain_Logistics/Data/'

# Detect the workbook by extension because the typed filename ("lgistics") did not match the real one
# "~$" prefix is Excel's temp lock file when the workbook is open, so it is excluded
xlsx_files = [f for f in os.listdir(data_dir) if f.lower().endswith('.xlsx') and not f.startswith('~$')]
print(xlsx_files)

# Fail early if there is not exactly one workbook in the folder
assert len(xlsx_files) == 1, f"Expected 1 .xlsx file in Data/, found: {xlsx_files}"
file_path = data_dir + xlsx_files[0]

expected_sheets = ['OrderList', 'FreightRates', 'WhCosts', 'WhCapacities', 'ProductsPerPlant', 'VmiCustomers', 'PlantPorts']

# sheet_name=None loads every sheet into a dict of DataFrames in one read
excel_sheets = pd.read_excel(file_path, sheet_name=None)
excel_sheets = {k.strip(): v for k, v in excel_sheets.items()}  # trim stray spaces in sheet names

# Fail early if a sheet is missing or renamed
assert all(s in excel_sheets for s in expected_sheets), f"Missing sheets: {set(expected_sheets) - set(excel_sheets)}"

# Lowercase dict keys so table names match the snake_case convention
raw_tables = {s.lower(): excel_sheets[s] for s in expected_sheets}
print(list(raw_tables.keys()))

['Supply chain logistics problem.xlsx']
['orderlist', 'freightrates', 'whcosts', 'whcapacities', 'productsperplant', 'vmicustomers', 'plantports']


In [2]:
# ---- Cell 2 code ----
raw_rows = {}  # keep original row counts for the reconciliation assert before export

# Baseline data quality: shape, null counts and exact duplicates per table
for name, df in raw_tables.items():
    raw_rows[name] = len(df)
    print(f'--- {name} | shape: {df.shape} ---')
    print(df.isna().sum()[df.isna().sum() > 0])  # show only columns that have nulls
    print(f'Exact duplicate rows: {df.duplicated().sum():,}')

--- orderlist | shape: (9215, 14) ---
Series([], dtype: int64)
Exact duplicate rows: 0
--- freightrates | shape: (1540, 11) ---
Series([], dtype: int64)
Exact duplicate rows: 3
--- whcosts | shape: (19, 2) ---
Series([], dtype: int64)
Exact duplicate rows: 0
--- whcapacities | shape: (19, 2) ---
Series([], dtype: int64)
Exact duplicate rows: 0
--- productsperplant | shape: (2036, 2) ---
Series([], dtype: int64)
Exact duplicate rows: 0
--- vmicustomers | shape: (14, 2) ---
Series([], dtype: int64)
Exact duplicate rows: 0
--- plantports | shape: (22, 2) ---
Series([], dtype: int64)
Exact duplicate rows: 0


In [3]:
# ---- Cell 3 code ----
# Make column names consistent (lowercase, underscores) so keys match across tables
for df in raw_tables.values():
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
    df.columns = df.columns.str.replace(r'[^a-z0-9_]+', '_', regex=True).str.strip('_')  # "cost/unit" -> "cost_unit"

id_cols = ['order_id', 'product_id', 'plant_code', 'plant_id']

for name, df in raw_tables.items():
    for col in id_cols:
        if col in df.columns:  # only touch ID columns that exist in this table
            # Cast to text, trim spaces, drop float artifacts like "101.0"
            df[col] = df[col].astype(str).str.strip().str.replace(r'\.0$', '', regex=True)
            df[col] = df[col].replace('nan', np.nan)  # restore true nulls so they are not counted as a real ID
    print(f'--- {name} | columns: {list(df.columns)} ---')

--- orderlist | columns: ['order_id', 'order_date', 'origin_port', 'carrier', 'tpt', 'service_level', 'ship_ahead_day_count', 'ship_late_day_count', 'customer', 'product_id', 'plant_code', 'destination_port', 'unit_quantity', 'weight'] ---
--- freightrates | columns: ['carrier', 'orig_port_cd', 'dest_port_cd', 'minm_wgh_qty', 'max_wgh_qty', 'svc_cd', 'minimum_cost', 'rate', 'mode_dsc', 'tpt_day_cnt', 'carrier_type'] ---
--- whcosts | columns: ['wh', 'cost_unit'] ---
--- whcapacities | columns: ['plant_id', 'daily_capacity'] ---
--- productsperplant | columns: ['plant_code', 'product_id'] ---
--- vmicustomers | columns: ['plant_code', 'customers'] ---
--- plantports | columns: ['plant_code', 'port'] ---


In [4]:
# ---- Cell 4 code ----
# Ports per table that must be populated for a record to be routable
port_cols = {
    'orderlist':    ['origin_port', 'destination_port'],
    'freightrates': ['orig_port_cd', 'dest_port_cd'],
}
anomaly_files = {
    'orderlist':    'anomaly_missing_order_ports.csv',
    'freightrates': 'anomaly_missing_lane_ports.csv',
}
anomalies = {}

for name, cols in port_cols.items():
    df = raw_tables[name]
    assert all(c in df.columns for c in cols), f"{name}: expected port columns not found"

    # Flag true nulls and blank strings, both make a route unusable
    blank_mask   = df[cols].astype(str).apply(lambda s: s.str.strip().eq('')).any(axis=1)
    missing_mask = df[cols].isna().any(axis=1) | blank_mask

    anomalies[name]    = df[missing_mask].copy()   # quarantined for review, not silently dropped
    raw_tables[name]   = df[~missing_mask].copy()  # only routable rows continue downstream
    print(f'--- {name} | anomalies: {len(anomalies[name]):,} | remaining: {len(raw_tables[name]):,} ---')

# Row conservation: nothing lost between raw and (clean + anomaly)
for name in port_cols:
    assert len(raw_tables[name]) + len(anomalies[name]) == raw_rows[name], f"{name}: rows lost during isolation"

--- orderlist | anomalies: 0 | remaining: 9,215 ---
--- freightrates | anomalies: 0 | remaining: 1,540 ---


In [5]:
# ---- Cell 5 code ----
def clean_table(df):
    df = df.copy()  # work on a copy so the raw data stays untouched

    # Convert date columns first, so they are not filled with text later
    for col in [c for c in df.columns if 'date' in c]:
        df[col] = pd.to_datetime(df[col], errors='coerce')  # bad values become NaT

    txt_cols = df.select_dtypes(include='object').columns

    # Trim text, then label missing text; numeric nulls stay NULL because 0 would distort rates and costs
    for col in txt_cols:
        df[col] = df[col].astype(str).str.strip().replace({'nan': np.nan, '': np.nan})
    df[txt_cols] = df[txt_cols].fillna('Not Specified')
    return df

clean_tables = {name: clean_table(df) for name, df in raw_tables.items()}

# Audit remaining nulls (numeric and dates only) per table
for name, df in clean_tables.items():
    print(f'--- {name} | shape: {df.shape} ---')
    print(df.isna().sum()[df.isna().sum() > 0])

--- orderlist | shape: (9215, 14) ---
Series([], dtype: int64)
--- freightrates | shape: (1540, 11) ---
Series([], dtype: int64)
--- whcosts | shape: (19, 2) ---
Series([], dtype: int64)
--- whcapacities | shape: (19, 2) ---
Series([], dtype: int64)
--- productsperplant | shape: (2036, 2) ---
Series([], dtype: int64)
--- vmicustomers | shape: (14, 2) ---
Series([], dtype: int64)
--- plantports | shape: (22, 2) ---
Series([], dtype: int64)


In [6]:
# ---- Cell 6 code ----
# Re-check duplicates after text trimming, since trimming can expose new exact matches
dup_counts = {name: df.duplicated().sum() for name, df in clean_tables.items()}
print(dup_counts)

# Drop only exact duplicates and keep the first copy of each row
for name, df in clean_tables.items():
    before = len(df)
    clean_tables[name] = df.drop_duplicates().reset_index(drop=True)
    print(f'--- {name} | removed: {before - len(clean_tables[name]):,} | remaining: {len(clean_tables[name]):,} ---')

# Only freightrates should lose rows, and none of the others should change
assert clean_tables['freightrates'].duplicated().sum() == 0, "freightrates: duplicates remain"
assert len(clean_tables['orderlist']) == 9215, "orderlist: row count changed"
assert len(clean_tables['freightrates']) == 1540 - dup_counts['freightrates'], "freightrates: unexpected row count"

print(f"Freight rows after dedup: {len(clean_tables['freightrates']):,}")

{'orderlist': np.int64(0), 'freightrates': np.int64(3), 'whcosts': np.int64(0), 'whcapacities': np.int64(0), 'productsperplant': np.int64(0), 'vmicustomers': np.int64(0), 'plantports': np.int64(0)}
--- orderlist | removed: 0 | remaining: 9,215 ---
--- freightrates | removed: 3 | remaining: 1,537 ---
--- whcosts | removed: 0 | remaining: 19 ---
--- whcapacities | removed: 0 | remaining: 19 ---
--- productsperplant | removed: 0 | remaining: 2,036 ---
--- vmicustomers | removed: 0 | remaining: 14 ---
--- plantports | removed: 0 | remaining: 22 ---
Freight rows after dedup: 1,537


In [7]:
# ---- Cell 7 code ----
# Validate before export
for name, df in clean_tables.items():
    txt_cols = df.select_dtypes(include='object').columns
    assert df[txt_cols].isna().sum().sum() == 0, f"{name}: text nulls remain"
    assert (df.columns == df.columns.str.lower()).all(), f"{name}: headers not lowercase"
    assert len(df) > 0, f"{name}: table is empty"

for name, cols in port_cols.items():
    assert clean_tables[name][cols].isna().sum().sum() == 0, f"{name}: null ports in clean table"
    assert (clean_tables[name][cols] != 'Not Specified').all().all(), f"{name}: filler port in clean table"

# One workbook with one sheet per clean table
output_file = data_dir + 'Clean_Supply_Chain_Logistics_Problem.xlsx'

# ExcelWriter keeps every sheet in a single file; index=False keeps the pandas row numbers out
with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    for name, df in clean_tables.items():
        df.to_excel(writer, sheet_name=name, index=False)
        print(f'--- {name} | sheet written | rows: {len(df):,} ---')

# Anomalies stay as separate CSVs so quarantined rows are never mixed with clean data
for name, df in anomalies.items():
    df.to_csv(f'{data_dir}{anomaly_files[name]}', index=False)
    print(f'--- {anomaly_files[name]} | rows: {len(df):,} ---')

# Re-read the workbook to confirm every sheet and row count survived the export
check_sheets = pd.read_excel(output_file, sheet_name=None)
assert set(check_sheets.keys()) == set(clean_tables.keys()), "Sheets missing in exported workbook"
for name, df in clean_tables.items():
    assert len(check_sheets[name]) == len(df), f"{name}: row count changed after export"

print(f'Saved: Clean_Supply_Chain_Logistics_Problem.xlsx | sheets: {len(check_sheets):,}')

--- orderlist | sheet written | rows: 9,215 ---
--- freightrates | sheet written | rows: 1,537 ---
--- whcosts | sheet written | rows: 19 ---
--- whcapacities | sheet written | rows: 19 ---
--- productsperplant | sheet written | rows: 2,036 ---
--- vmicustomers | sheet written | rows: 14 ---
--- plantports | sheet written | rows: 22 ---
--- anomaly_missing_order_ports.csv | rows: 0 ---
--- anomaly_missing_lane_ports.csv | rows: 0 ---
Saved: Clean_Supply_Chain_Logistics_Problem.xlsx | sheets: 7


In [8]:
# ---- Cell 8 code ----
# MySQL Workbench import wizard reads CSV, not xlsx, so export each clean table separately
for name, df in clean_tables.items():
    df.to_csv(f'{data_dir}clean_{name}.csv', index=False)  # index=False keeps pandas row numbers out
    print(f'--- clean_{name}.csv | rows: {len(df):,} ---')

# Re-read each CSV to confirm the row counts survived the export
for name, df in clean_tables.items():
    assert len(pd.read_csv(f'{data_dir}clean_{name}.csv')) == len(df), f"{name}: row count changed after export"

print(f'Tables exported: {len(clean_tables):,}')

--- clean_orderlist.csv | rows: 9,215 ---
--- clean_freightrates.csv | rows: 1,537 ---
--- clean_whcosts.csv | rows: 19 ---
--- clean_whcapacities.csv | rows: 19 ---
--- clean_productsperplant.csv | rows: 2,036 ---
--- clean_vmicustomers.csv | rows: 14 ---
--- clean_plantports.csv | rows: 22 ---
Tables exported: 7
